# Urban Heat & Cooling-Priority Mapping — Land-Change Diagnostic

**NUS-ISS Practice Module, Week 2.** Checks whether the 6-year composite
window (2021-2026) is blending genuinely different land-cover states in any
subzone — e.g. new development, reclaimed land, cleared vegetation — rather
than averaging noise around a stable value. A `.median()` composite doesn't
distinguish those cases; if land changed partway through the window, the
result is a biased mix of two real states, not a clean average, and that
bias lands specifically in the subzones most likely to matter for a
cooling-priority tool (redevelopment zones).

Uses **Dynamic World** (already a locked data source, currently
validation-only for S3) to compare the dominant land-cover class between
the first two years of the window and the last two, per subzone. Subzones
with a high fraction of pixels that switched class get flagged as
lower-confidence, rather than silently trusted at face value alongside
every other subzone.

One notebook, run top to bottom:

1. **Setup** — install deps, authenticate Earth Engine, mount Drive
2. **LC.1** — config
3. **LC.2** — fetch URA subzones
4. **LC.3** — build early-period and late-period dominant-class composites
5. **LC.4** — pixel-level change mask
6. **LC.5** — zonal change fraction per subzone
7. **LC.6** — match subzone_id to heat-variants CSV
8. **LC.7** — flag unstable subzones
9. **LC.8** — verdict
10. **LC.9** — save to Drive

Run cells in order.

---


# SETUP — run once per session

## Setup 1 — Install dependencies

In [1]:
# --- SETUP CELL 1: Install all dependencies used in this notebook ----------
!pip install -q earthengine-api pandas requests


## Setup 2 — Authenticate & initialize Earth Engine

In [2]:
# --- SETUP CELL 2: Authenticate & initialize Earth Engine -------------------
import ee

PROJECT_ID = "nus-iss-urban-heat-sg"  # <-- your GCP project

if PROJECT_ID == "your-gcp-project-id":
    raise ValueError("PROJECT_ID is still the placeholder. Set it, then re-run this cell.")

try:
    ee.Initialize(project=PROJECT_ID)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT_ID)

print("EE initialized OK, project:", PROJECT_ID)


EE initialized OK, project: nus-iss-urban-heat-sg


## Setup 3 — Mount Google Drive

In [3]:
# --- SETUP CELL 3: Mount Google Drive ----------------------------------------
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted at /content/drive")


Mounted at /content/drive
Drive mounted at /content/drive


## Setup 4 — Initialize shared results tracker

In [4]:
# --- SETUP CELL 4: Initialize shared results tracker ------------------------
lc_results = {}
print("lc_results initialized — populated by the LC.8 verdict cell.")


lc_results initialized — populated by the LC.8 verdict cell.


---
# LC — Land-Change Diagnostic (Dynamic World, early vs. late years)


## LC.1 — Config

Two non-overlapping 3-year windows: 2018-2021 vs. 2022-2025. Confirmed via
LC.1b that 1-year windows don't have enough raw Dynamic World imagery over
Singapore to trust a mode-based comparison (only 10-13 images/year, before
any filtering) — widening to 3 years each fixes the sample-size problem
directly, the way widening `gee_heat_variants.ipynb`'s season window fixed
its scene-count problem earlier. No season-month restriction: LC.1b also
confirmed that alone wasn't enough to fix the sparsity, and land-cover
*class* (unlike LST) isn't seasonally distorted the way temperature is, so
there's no real downside to using full years here.


In [5]:
# --- LC CELL 1: Config -------------------------------------------------------
sg_bbox = ee.Geometry.Rectangle([103.55, 1.15, 104.10, 1.48])

EARLY_START = "2018-01-01"
EARLY_END = "2021-01-01"    # 3-year window
LATE_START = "2022-01-01"
LATE_END = "2025-01-01"     # 3-year window — non-overlapping with early, both fully
                             # in the past as of this analysis (Jul 2026)

DRY_SEASON_MONTHS = [4, 5, 10, 11]  # NOT applied to the main early/late comparison
                                      # (see LC.1b) — kept here only so LC.1b's
                                      # season-filter sensitivity test still has
                                      # something to test against if re-run later

CHANGE_FRACTION_THRESHOLD = 0.15  # subzones with >15% of pixels switching
                                    # dominant class get flagged as unstable —
                                    # a judgment call, not a validated cutoff;
                                    # revisit if the flagged count looks off

EXPORT_FOLDER = "urban_heat_sg"
HEAT_CSV_PATH = f"/content/drive/MyDrive/{EXPORT_FOLDER}/heat_variants_subzone.csv"
OUT_PATH = f"/content/drive/MyDrive/{EXPORT_FOLDER}/land_change_flags.csv"

SUBZONE_DATASET_ID = "d_8594ae9ff96d0c708bc2af633048edfb"
SUBZONE_LOCAL_PATH = "/content/ura_subzones_lc.geojson"
SUBZONE_ID_PROPERTY = "SUBZONE_N"
TARGET_SCALE = 10

import os
if not os.path.exists(HEAT_CSV_PATH):
    raise FileNotFoundError(f"{HEAT_CSV_PATH} not found. Run gee_heat_variants.ipynb's export first.")
print("✅ Heat-variants CSV found.")
print(f"Comparing {EARLY_START} to {EARLY_END}  vs.  {LATE_START} to {LATE_END}")


✅ Heat-variants CSV found.
Comparing 2018-01-01 to 2021-01-01  vs.  2022-01-01 to 2025-01-01


## LC.1b — Sanity check: does the season filter actually cause the sparsity?

Direct A/B test, same AOI and same rolling windows, with vs. without the
`DRY_SEASON_MONTHS` restriction — isolates whether the season filter is the
cause of the near-zero image counts before changing anything in the main
pipeline. Doesn't affect LC.2 onward; safe to run and inspect first.


In [6]:
# --- LC CELL 1b: Season-filter sensitivity test ------------------------------
dw_test = ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1").filterBounds(sg_bbox).select("label")

def season_filter_test(months):
    return ee.Filter.Or(*[ee.Filter.calendarRange(m, m, "month") for m in months])

early_full = dw_test.filterDate(EARLY_START, EARLY_END)
early_season = early_full.filter(season_filter_test(DRY_SEASON_MONTHS))
late_full = dw_test.filterDate(LATE_START, LATE_END)
late_season = late_full.filter(season_filter_test(DRY_SEASON_MONTHS))

n_early_full = early_full.size().getInfo()
n_early_season = early_season.size().getInfo()
n_late_full = late_full.size().getInfo()
n_late_season = late_season.size().getInfo()

print("Season-filter sensitivity test:")
print(f"  Early window ({EARLY_START} to {EARLY_END}):")
print(f"    full 12 months:     {n_early_full}")
print(f"    season-restricted:  {n_early_season}"
      + (f"  ({100*n_early_season/n_early_full:.1f}% of full)" if n_early_full else "  (full count is 0)"))
print(f"  Late window ({LATE_START} to {LATE_END}):")
print(f"    full 12 months:     {n_late_full}")
print(f"    season-restricted:  {n_late_season}"
      + (f"  ({100*n_late_season/n_late_full:.1f}% of full)" if n_late_full else "  (full count is 0)"))

print()
season_is_main_cause = True
if n_early_full > 0 and n_early_season / n_early_full < 0.5:
    print("⚠️  Season restriction cuts the EARLY window's image count by more than half —")
    print("   consistent with the season filter being a major cause of the sparsity.")
elif n_early_full > 0:
    season_is_main_cause = False

if n_late_full > 0 and n_late_season / n_late_full < 0.5:
    print("⚠️  Season restriction cuts the LATE window's image count by more than half —")
    print("   consistent with the season filter being a major cause of the sparsity.")
elif n_late_full > 0:
    season_is_main_cause = False

if n_early_full < 20 or n_late_full < 20:
    print("\n⚠️  Even the FULL 12-month count is low in at least one window — this points to a")
    print("   deeper Dynamic World coverage gap for this AOI/period, not just the season filter.")
    print("   If so, dropping the season restriction alone will NOT fully fix it — investigate")
    print("   the full-year count further before assuming a simple fix.")
    season_is_main_cause = False

print()
if season_is_main_cause:
    print("✅ Conclusion: the season filter is the dominant cause. Dropping DRY_SEASON_MONTHS")
    print("   for this check (keeping the same rolling windows) should resolve the sparsity.")
else:
    print("⚠️  Conclusion: the season filter is NOT the whole story. Full 12-month counts are")
    print("   also thin in at least one window — dropping the season filter will help, but may")
    print("   not be sufficient on its own. Report the full_early/full_late numbers above before")
    print("   deciding next steps.")


Season-filter sensitivity test:
  Early window (2018-01-01 to 2021-01-01):
    full 12 months:     64
    season-restricted:  21  (32.8% of full)
  Late window (2022-01-01 to 2025-01-01):
    full 12 months:     43
    season-restricted:  16  (37.2% of full)

⚠️  Season restriction cuts the EARLY window's image count by more than half —
   consistent with the season filter being a major cause of the sparsity.
⚠️  Season restriction cuts the LATE window's image count by more than half —
   consistent with the season filter being a major cause of the sparsity.

✅ Conclusion: the season filter is the dominant cause. Dropping DRY_SEASON_MONTHS
   for this check (keeping the same rolling windows) should resolve the sparsity.


## LC.2 — Fetch URA subzones

In [7]:
# --- LC CELL 2: Fetch URA subzones --------------------------------------------
import requests
import json

def fetch_datagovsg_geojson(dataset_id, out_path):
    poll_url = f"https://api-open.data.gov.sg/v1/public/api/datasets/{dataset_id}/poll-download"
    r = requests.get(poll_url)
    r.raise_for_status()
    payload = r.json()
    if payload.get("code") != 0:
        raise RuntimeError(f"data.gov.sg API error: {payload.get('errMsg')}")
    download_url = payload["data"]["url"]
    geojson_bytes = requests.get(download_url).content
    with open(out_path, "wb") as f:
        f.write(geojson_bytes)
    return out_path

subzone_path = fetch_datagovsg_geojson(SUBZONE_DATASET_ID, SUBZONE_LOCAL_PATH)
with open(subzone_path) as f:
    subzone_geojson = json.load(f)

for feature in subzone_geojson.get("features", []):
    props = feature.get("properties", {})
    for old_key in list(props.keys()):
        if "." in old_key:
            props[old_key.replace(".", "_")] = props.pop(old_key)

subzones = ee.FeatureCollection(subzone_geojson)
n_subzones_total = subzones.size().getInfo()
print(f"Loaded {n_subzones_total} subzones")

sample_props = subzones.first().propertyNames().getInfo()
if SUBZONE_ID_PROPERTY not in sample_props:
    raise ValueError(f"'{SUBZONE_ID_PROPERTY}' not found. Candidates: {sample_props}")
print(f"✅ SUBZONE_ID_PROPERTY = '{SUBZONE_ID_PROPERTY}' confirmed present.")


Loaded 332 subzones
✅ SUBZONE_ID_PROPERTY = 'SUBZONE_N' confirmed present.


## LC.3 — Early vs. late dominant land-cover class

Dynamic World provides a per-pixel class label at ~10m, roughly Sentinel-2
cadence. `ee.Reducer.mode()` takes the most frequently observed class per
pixel across all images in each period — the "typical" class for that
period, robust to a single misclassified scene.


In [8]:
# --- LC CELL 3: Early/late dominant-class composites --------------------------
# No DRY_SEASON_MONTHS restriction here — confirmed via LC.1b that it isn't the
# main constraint (full 12-month counts were also too thin) and land-cover class
# isn't seasonally distorted the way LST is, so full years are used directly.
dw = ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1").filterBounds(sg_bbox).select("label")

dw_early = dw.filterDate(EARLY_START, EARLY_END)
dw_late = dw.filterDate(LATE_START, LATE_END)

n_early = dw_early.size().getInfo()
n_late = dw_late.size().getInfo()
print(f"Dynamic World images — early period ({EARLY_START} to {EARLY_END}): {n_early}")
print(f"Dynamic World images — late period ({LATE_START} to {LATE_END}): {n_late}")

if n_early == 0 or n_late == 0:
    raise RuntimeError("Zero images in one period — cannot compute a meaningful change comparison. "
                        "Check EARLY_START/EARLY_END/LATE_START/LATE_END.")
if n_early < 20 or n_late < 20:
    print("⚠️  Low image count in at least one period — the 'dominant class' mode may be noisy. "
          "Treat the resulting change flags as indicative, not definitive.")

early_class = dw_early.reduce(ee.Reducer.mode()).rename("early_class").clip(sg_bbox)
late_class = dw_late.reduce(ee.Reducer.mode()).rename("late_class").clip(sg_bbox)
print("Early/late dominant-class composites built.")


Dynamic World images — early period (2018-01-01 to 2021-01-01): 64
Dynamic World images — late period (2022-01-01 to 2025-01-01): 43
Early/late dominant-class composites built.


## LC.4 — Pixel-level change mask

In [9]:
# --- LC CELL 4: Change mask ---------------------------------------------------
changed_mask = early_class.neq(late_class).rename("changed")
print("Change mask built (1 = dominant class differs between early and late periods).")


Change mask built (1 = dominant class differs between early and late periods).


## LC.4b — Overall map-wide change (headline number)

A single number for the whole map: what fraction of Singapore's land area
shows a different dominant land-cover class between the two periods. No
subzone boundaries, no threshold, no judgment call — just the direct
island-wide answer to "how different is 2022-2025 from 2018-2021?"


In [10]:
# --- LC CELL 4b: Overall map-wide change fraction -----------------------------
overall_stats = changed_mask.reduceRegion(
    reducer=ee.Reducer.mean().combine(ee.Reducer.count(), sharedInputs=True),
    geometry=sg_bbox,
    scale=TARGET_SCALE,
    maxPixels=1e10,
    bestEffort=True,
)
overall_change_fraction = overall_stats.get("changed_mean").getInfo()
overall_pixel_count = overall_stats.get("changed_count").getInfo()

print(f"Overall map-wide change: {overall_change_fraction*100:.1f}% of {overall_pixel_count:,} valid "
      f"pixels across Singapore show a different dominant land-cover class between "
      f"{EARLY_START}-{EARLY_END} and {LATE_START}-{LATE_END}.")
print()
print("This is a single island-wide figure — it does not tell you WHERE the change is")
print("concentrated (that's what the per-subzone flags below are for), only how much of")
print("the total land area changed overall.")


Overall map-wide change: 7.1% of 22,497,105 valid pixels across Singapore show a different dominant land-cover class between 2018-01-01-2021-01-01 and 2022-01-01-2025-01-01.

This is a single island-wide figure — it does not tell you WHERE the change is
concentrated (that's what the per-subzone flags below are for), only how much of
the total land area changed overall.


## LC.5 — Zonal change fraction per subzone

In [11]:
# --- LC CELL 5: Zonal change fraction ------------------------------------------
# Diagnostic image with 3 bands: the change mask itself, plus each period's raw
# class band (used only for its valid-pixel COUNT, not its class values) — lets
# one reduceRegions call report change_fraction AND per-period pixel availability,
# so a coverage gap can be pinned to "early period", "late period", or "both"
# instead of just showing up as one opaque combined low-pixel-count number.
diagnostic_img = changed_mask.rename("changed").addBands(early_class.rename("early")).addBands(late_class.rename("late"))

zonal = diagnostic_img.reduceRegions(
    collection=subzones,
    reducer=ee.Reducer.mean().combine(ee.Reducer.count(), sharedInputs=True),
    scale=TARGET_SCALE,
    tileScale=4,
)

records = zonal.map(
    lambda f: ee.Feature(None, {
        "subzone_id": f.get(SUBZONE_ID_PROPERTY),
        "change_fraction": f.get("changed_mean"),
        "n_valid_pixels": f.get("changed_count"),      # valid in BOTH periods (what the change comparison uses)
        "n_valid_pixels_early": f.get("early_count"),  # valid in early period only
        "n_valid_pixels_late": f.get("late_count"),    # valid in late period only
    })
).getInfo()

import pandas as pd
lc_df = pd.DataFrame([r["properties"] for r in records["features"]])
n_null = lc_df["change_fraction"].isna().sum()
print(f"Zonal reduction: {len(lc_df)} subzones, {n_null} with null (no valid pixels in either period).")
print(lc_df[["n_valid_pixels", "n_valid_pixels_early", "n_valid_pixels_late"]].describe().to_string())

# Pinpoint WHERE the gap is for subzones with zero combined pixels.
zero_combined = lc_df[lc_df["n_valid_pixels"].fillna(0) == 0]
if len(zero_combined) > 0:
    early_zero = (zero_combined["n_valid_pixels_early"].fillna(0) == 0).sum()
    late_zero = (zero_combined["n_valid_pixels_late"].fillna(0) == 0).sum()
    both_zero = ((zero_combined["n_valid_pixels_early"].fillna(0) == 0) & (zero_combined["n_valid_pixels_late"].fillna(0) == 0)).sum()
    print(f"\nOf {len(zero_combined)} subzones with zero combined valid pixels:")
    print(f"  {early_zero} have zero pixels in the EARLY period ({EARLY_START} to {EARLY_END})")
    print(f"  {late_zero} have zero pixels in the LATE period ({LATE_START} to {LATE_END})")
    print(f"  {both_zero} have zero pixels in BOTH periods")
    print("  (early-only or late-only zero => a period-specific coverage gap, not a general one)")

lc_df_with_data = lc_df.dropna(subset=["change_fraction"])
print("\nTop 10 by change_fraction (among subzones with data):")
print(lc_df_with_data.sort_values("change_fraction", ascending=False).head(10).to_string(index=False))


Zonal reduction: 332 subzones, 0 with null (no valid pixels in either period).
       n_valid_pixels  n_valid_pixels_early  n_valid_pixels_late
count      332.000000            332.000000           332.000000
mean     24177.530120          24177.530120         24177.530120
std      65276.940042          65276.940042         65276.940042
min        454.000000            454.000000           454.000000
25%       6496.250000           6496.250000          6496.250000
50%      12310.500000          12310.500000         12310.500000
75%      21090.000000          21090.000000         21090.000000
max     692960.000000         692960.000000        692960.000000

Top 10 by change_fraction (among subzones with data):
 change_fraction  n_valid_pixels  n_valid_pixels_early  n_valid_pixels_late          subzone_id
        0.754728            8073                  8073                 8073              GARDEN
        0.538300           23296                 23296                23296              

## LC.6 — Match subzone_id to heat-variants CSV

In [12]:
# --- LC CELL 6: Match subzone_id ----------------------------------------------
heat = pd.read_csv(HEAT_CSV_PATH)
heat_ids = set(heat["subzone_id"].astype(str))
lc_ids = set(lc_df_with_data["subzone_id"].astype(str))

n_matched = len(heat_ids & lc_ids)
unmatched_heat = sorted(heat_ids - lc_ids)
print(f"Matched: {n_matched} / {len(heat_ids)} heat-CSV subzones")
if unmatched_heat:
    print(f"\n⚠️  {len(unmatched_heat)} heat-CSV subzones have no change-fraction value "
          f"(no valid Dynamic World pixels in one period). Full list — eyeball these for")
    print(f"   plausibility (expect mostly water/offshore/tiny subzones; anything else is")
    print(f"   worth investigating further, not just assuming benign):")
    for name in unmatched_heat:
        print(f"     {name}")

lc_df_with_data = lc_df_with_data[lc_df_with_data["subzone_id"].astype(str).isin(heat_ids)].copy()


Matched: 332 / 332 heat-CSV subzones


## LC.7 — Flag unstable subzones


In [13]:
# --- LC CELL 7: Flag unstable subzones + explicit status for ALL subzones ----
MIN_VALID_PIXELS = 30  # below this, the mode comparison is likely noise, not signal

lc_df_with_data["land_change_flag"] = (
    (lc_df_with_data["change_fraction"] > CHANGE_FRACTION_THRESHOLD)
    & (lc_df_with_data["n_valid_pixels"] >= MIN_VALID_PIXELS)
)
lc_df_with_data["low_sample_size"] = lc_df_with_data["n_valid_pixels"] < MIN_VALID_PIXELS

# Build a full-coverage table: every heat-CSV subzone gets an explicit status,
# including the ones with zero valid Dynamic World pixels — these must NOT
# silently disappear from the output, since a downstream join would otherwise
# treat "we don't know" the same as "confirmed stable", which is a materially
# different and false claim.
full = pd.DataFrame({"subzone_id": sorted(heat_ids)})
full = full.merge(lc_df_with_data[["subzone_id", "change_fraction", "n_valid_pixels", "n_valid_pixels_early",
                                    "n_valid_pixels_late", "land_change_flag", "low_sample_size"]],
                   on="subzone_id", how="left")

def status(row):
    if pd.isna(row["change_fraction"]):
        return "insufficient_data"
    if row["low_sample_size"]:
        return "insufficient_data"
    if row["land_change_flag"]:
        return "flagged"
    return "stable"

full["status"] = full.apply(status, axis=1)

counts = full["status"].value_counts()
print("Status breakdown (all", len(full), "heat-CSV subzones):")
print(counts.to_string())

if counts.get("flagged", 0) > 0:
    print("\nFlagged subzones (change > threshold AND enough pixels to trust it), highest first:")
    print(full[full["status"] == "flagged"].sort_values("change_fraction", ascending=False)
          [["subzone_id", "change_fraction", "n_valid_pixels"]].to_string(index=False))
    print("\n⚠️  The LST composite for these subzones may be blending genuinely different land")
    print("   states across the window, not just averaging noise. Treat as lower-confidence.")

if counts.get("insufficient_data", 0) > 0:
    print(f"\nℹ️  {counts.get('insufficient_data', 0)} subzones have status 'insufficient_data' —")
    print("   NOT the same as 'stable'. Don't treat these as confirmed unchanged; the honest")
    print("   answer for them is 'unknown', and that should stay visible downstream.")


Status breakdown (all 332 heat-CSV subzones):
status
stable     286
flagged     46

Flagged subzones (change > threshold AND enough pixels to trust it), highest first:
              subzone_id  change_fraction  n_valid_pixels
                  GARDEN         0.754728            8073
                  TENGEH         0.538300           23296
                    PARK         0.506329           15046
           SENGKANG WEST         0.451408           17099
                   WENYA         0.396929           20263
     LAKESIDE (BUSINESS)         0.369088           11746
          CHANGI AIRPORT         0.362275          360861
     TUAS VIEW EXTENSION         0.359169          281832
              PLANTATION         0.347201            9336
          TUAS PROMENADE         0.325347           42579
            STRAITS VIEW         0.280394           11681
               CLEANTECH         0.273761            7252
                PATERSON         0.272714            1848
      LAKESIDE (LEIS

## LC.8 — Verdict

In [14]:
# --- LC CELL 8: Verdict --------------------------------------------------------
print("\n--- LC Verdict ---")

n_flagged_final = (full["status"] == "flagged").sum()
n_insufficient = (full["status"] == "insufficient_data").sum()
pct_insufficient = 100 * n_insufficient / len(full)

lc_checks = {
    "Dynamic World images found in both periods": n_early > 0 and n_late > 0,
    "Every heat-CSV subzone has an explicit status": full["status"].notna().all() and len(full) == len(heat_ids),
    "insufficient_data isn't the majority (<50%)": pct_insufficient < 50,
}

for check, passed in lc_checks.items():
    print(f"  [{'PASS' if passed else 'FAIL'}] {check}")

lc_pass = all(lc_checks.values())
print(f"\n{n_flagged_final} flagged, {n_insufficient} insufficient_data ({pct_insufficient:.1f}%), "
      f"{len(full) - n_flagged_final - n_insufficient} stable — out of {len(full)} total.")
if lc_pass:
    print("\n✅ LC PASS: land_change_flags.csv ready to save, with full subzone coverage.")
else:
    print("\n⚠️  LC FAIL: resolve flagged step(s) above first.")

lc_results["LC_land_change"] = {
    "status": "PASS" if lc_pass else "FAIL",
    "checks": lc_checks,
    "n_flagged": int(n_flagged_final),
    "n_insufficient_data": int(n_insufficient),
    "pct_insufficient_data": pct_insufficient,
    "threshold": CHANGE_FRACTION_THRESHOLD,
    "min_valid_pixels": MIN_VALID_PIXELS,
}



--- LC Verdict ---
  [PASS] Dynamic World images found in both periods
  [PASS] Every heat-CSV subzone has an explicit status
  [PASS] insufficient_data isn't the majority (<50%)

46 flagged, 0 insufficient_data (0.0%), 286 stable — out of 332 total.

✅ LC PASS: land_change_flags.csv ready to save, with full subzone coverage.


## LC.9 — Save to Drive

In [15]:
# --- LC CELL 9: Save to Drive ---------------------------------------------------
out = full[["subzone_id", "change_fraction", "n_valid_pixels", "n_valid_pixels_early",
            "n_valid_pixels_late", "status"]]
out.to_csv(OUT_PATH, index=False)
print(f"Saved: {OUT_PATH} ({len(out)} subzones, ALL heat-CSV subzones covered)")
print("\nEvery subzone has a 'status': 'flagged', 'stable', or 'insufficient_data'.")
print("n_valid_pixels_early/late let you check WHICH period is thin for any given subzone.")
print("Join on subzone_id. Treat 'insufficient_data' as genuinely unknown, not as 'stable' —")
print("don't let a downstream left-join silently coerce missing rows into a false negative.")


Saved: /content/drive/MyDrive/urban_heat_sg/land_change_flags.csv (332 subzones, ALL heat-CSV subzones covered)

Every subzone has a 'status': 'flagged', 'stable', or 'insufficient_data'.
n_valid_pixels_early/late let you check WHICH period is thin for any given subzone.
Join on subzone_id. Treat 'insufficient_data' as genuinely unknown, not as 'stable' —
don't let a downstream left-join silently coerce missing rows into a false negative.
